# Stopping time: policy evaluation by simulation

A port of `stoppingtimeSimPI.mlx`.

The asset-selling problem of Unit 10, but asking a different question. Unit
10 looked for the *optimal* threshold. Here a threshold is **fixed** -- sell
whenever the scaled offer is at least $c_{25}$ -- and we evaluate what that
policy is worth, from simulated offers alone.

This is the policy-evaluation half of policy iteration, in the smallest
setting where it can be seen. Because the policy is fixed there is no
minimisation anywhere: the value $\theta$ of waiting satisfies the *linear*
relation

$$\theta=\bar\alpha\,\mathbb{E}\big[h(w)\big], \qquad h(w)=\begin{cases}w & \text{if } w\geq c_M\\ \theta & \text{otherwise,}\end{cases}$$

and the notebook drives it by stochastic approximation. Since the relation is
linear in $\theta$ it can also be solved in closed form, which gives an exact
answer to check against:

$$\theta^\star=\frac{\bar\alpha\,\bar\theta}{1-\bar\alpha\gamma}, \qquad
\gamma=\sum_{i<M}p_i, \qquad \bar\theta=\sum_{i\geq M}p_ic_i .$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)
plt.rcParams["figure.figsize"] = (14, 4)

from policy_iteration import sample_dist

rng = np.random.default_rng(0)

In [ ]:
C        = 101
alpha    = 0.5          # step size
alphabar = 0.99         # discount factor
c        = 10000 + np.arange(C)*100
var      = 2000.0**2
p        = np.exp(-1/(2*var)*(c - c[C//2])**2)
p        = p/p.sum()
M        = 25           # the fixed threshold: sell at or above c[M-1]

print(f"threshold: sell at or above {c[M-1]} euros")

In [ ]:
Niter = 100_000
theta = np.zeros(Niter+1); x = np.zeros(Niter+1)
theta[0] = c[C//2]; x[0] = c[C//2]

for ell in range(1, Niter+1):
    x[ell] = c[sample_dist(p, rng)]
    gamma_t = alpha if ell <= 1000 else alpha/(ell/1000)
    # the fixed policy: take the offer if it clears the threshold, else wait
    h = x[ell-1] if x[ell-1] >= c[M-1] else theta[ell-1]
    theta[ell] = (1 - gamma_t)*theta[ell-1] + gamma_t*alphabar*h

# closed form, for comparison
gamma      = p[:M-1].sum()
thetabar   = float(p[M-1:] @ c[M-1:])
theta_star = alphabar*thetabar/(1 - alphabar*gamma)

print(f"simulated theta  -> {theta[-1]:.1f}")
print(f"closed form      -> {theta_star:.1f}")
print(f"relative error     {abs(theta[-1]-theta_star)/theta_star:.2%}")
print(f"\nP(offer below the threshold) = {gamma:.4f}")

In [ ]:
fig, ax = plt.subplots(1, 2)
ax[0].plot(theta); ax[0].axhline(theta_star, color="r", ls="--", lw=1)
ax[0].set_xlabel("iteration number"); ax[0].set_ylabel(r"$\theta$")
ax[1].plot(x[1:], ",")
ax[1].axhline(c[M-1], color="r", ls="--", lw=1)
ax[1].set_xlabel("iteration number"); ax[1].set_ylabel("sampled offer")
plt.tight_layout(); plt.show()

Simulation lands within a fraction of a percent of the closed form.

Two things are worth drawing out. First, this policy is worth about
$15\,160$, while Unit 10's *optimal* policy -- threshold $16\,900$ -- is
worth $16\,858$. Evaluating a policy and improving it are genuinely separate
operations, and this notebook does only the first. Running the improvement
step on top of it is what policy iteration would do next.

Second, the closed form exists only because the policy is fixed. The moment
a $\min$ or $\max$ enters, the relation stops being linear and there is no
such formula -- which is exactly why policy *evaluation* is the cheap half
of policy iteration.